# ANDES basic usage for the IEEE 14-bus SG–GFL case

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xuwkk/Neural-SmallSignal-UC-Tutorial/blob/main/small_signal_concept_with_andes.ipynb)

This notebook shows the minimum workflow used by this tutorial:

1. locate the built-in ANDES Excel case;
2. inspect static and dynamic model tables;
3. run an AC power flow and read solved quantities;
4. evaluate one small-signal operating point with the project wrapper.

In an ANDES workbook, each worksheet represents one model type and each row represents one device.

### Google Colab setup

> **GOOGLE COLAB ONLY:** The next cell clones the complete project and installs the Colab dependencies. It does nothing in a local Jupyter environment.

In [ ]:
# GOOGLE COLAB ONLY: clone the project and install its dependencies.
import os
import subprocess
import sys
from pathlib import Path

IN_GOOGLE_COLAB = "google.colab" in sys.modules
if IN_GOOGLE_COLAB:
    COLAB_REPOSITORY_URL = "https://github.com/xuwkk/Neural-SmallSignal-UC-Tutorial.git"
    COLAB_PROJECT_DIR = Path("/content/Neural-SmallSignal-UC-Tutorial")

    if not COLAB_PROJECT_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", COLAB_REPOSITORY_URL, str(COLAB_PROJECT_DIR)],
            check=True,
        )

    os.chdir(COLAB_PROJECT_DIR)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
        check=True,
    )
    print(f"Colab project directory: {COLAB_PROJECT_DIR}")

In [ ]:
import pandas as pd
from IPython.display import display

import system_constants as const
import small_signal_functions as ss
import andes

print(f"ANDES version: {andes.__version__}")
print(f"Case file: {ss.get_case_path()}")

## 1. Inspect the Excel workbook

The installed workbook can be opened in Excel, but it should be treated as read-only. Copy it into the project before making permanent edits. Pandas can list the worksheets without modifying the file.

In [ ]:
sheet_names = pd.ExcelFile(ss.get_case_path()).sheet_names
print("List of the worksheets (models)")
print(sheet_names)

## 2. Load the case and inspect ANDES model tables

`system.<Model>.as_df()` returns the input data for one ANDES model as a Pandas DataFrame. Important static models are `Bus`, `PQ`, `PV`, `Slack`, and `Line`. Important dynamic models in this case include `GENROU`, `TGOV1`, `REGCP1`, `REECA1`, `REPCA1`, and `PLL1`.

In [ ]:
# Load the case via andes
system = andes.load(
    str(ss.get_case_path()),
    no_output=True,
    pycode_path=str(const.CACHE_DIR / "andes_pycode"),
)

# Important static and dynamic models
model_names = [
    "Bus", "PQ", "PV", "Slack", "Line",
    "GENROU", "TGOV1", "REGCP1", "PLL1",
    "REECA1", "REPCA1",
]

model_counts = pd.DataFrame(
    {"model": name, "device_count": getattr(system, name).n}
    for name in model_names
)
display(model_counts)

In [ ]:
def show_table(model_name):
    """Return one non-empty ANDES model table."""
    model = getattr(system, model_name)
    if model.n == 0:
        raise ValueError(f"{model_name} has no devices in this case.")
    return model.as_df()

# Change "PV" to another model name to inspect a different table.
display(show_table("PV"))

### Static-to-dynamic links

The `gen` column in `GENROU` points to a `PV` or `Slack` index. The `syn` column in a governor or exciter points to a synchronous-machine index. For the grid-following converter, `REGCP1.gen` points to the static `PV`, while its PLL reference points to `PLL1`.

In [ ]:
for model_name in ["GENROU", "TGOV1", "REGCP1", "PLL1"]:
    print(f"{model_name}:")
    display(show_table(model_name))

## 3. Run AC power flow and read solved values

`as_df()` shows model inputs such as `p0` and `q0`. After `PFlow.run()`, solved algebraic variables are stored in arrays ending in `.v`, such as `Bus.v.v`, `PV.p.v`, and `Slack.p.v`. Angles below are reported in radians.

In [ ]:
if not system.PFlow.run():
    raise RuntimeError("ANDES power flow did not converge.")

bus_results = pd.DataFrame({
    "bus": system.Bus.idx.v,
    "voltage_pu": system.Bus.v.v,
    "angle_rad": system.Bus.a.v,
})
display(bus_results)

In [ ]:
pv_results = pd.DataFrame({
    "pv_idx": system.PV.idx.v,
    "bus": system.PV.bus.v,
    "status": system.PV.u.v,
    "specified_p_mw": system.PV.p0.v * const.SYSTEM_BASE_MVA,
    "solved_p_mw": system.PV.p.v * const.SYSTEM_BASE_MVA,
    "solved_q_mvar": system.PV.q.v * const.SYSTEM_BASE_MVA,
})

slack_results = pd.DataFrame({
    "slack_idx": system.Slack.idx.v,
    "bus": system.Slack.bus.v,
    "solved_p_mw": system.Slack.p.v * const.SYSTEM_BASE_MVA,
    "solved_q_mvar": system.Slack.q.v * const.SYSTEM_BASE_MVA,
})

display(pv_results)
display(slack_results)

## 4. Evaluate one project operating point

The project wrapper sends a complete bus-level load vector to ANDES in `LOAD_BUSES` order, and each reactive load is derived from its base-case Q/P ratio. In the simplified UC workflow, this vector is reconstructed from total load using fixed base-case shares. The SG order is buses `(2, 3, 6)`. The Slack SG at bus 1 remains online and balances demand plus AC losses.

In [ ]:
print("Load bus order:", const.LOAD_BUSES)

point = ss.OperatingPoint(
    sg_online=(1, 1, 1),
    sg_power_mw=(40, 40, 30),
    gfl_power_mw=35,
    load_power_mw=tuple(const.BASE_LOAD_POWER_MW),
)

with ss.silence_andes_output():
    dynamic_system = ss.build_system(point)
    slack_power_mw = ss.solve_power_flow(dynamic_system)
    result = ss.calculate_small_signal(dynamic_system, slack_power_mw)
pd.Series({
    "label (0 = stable)": result.label,
    "critical eigenvalue real (1/s)": result.critical_real,
    "critical eigenvalue imag (1/s)": result.critical_imag,
    "slack power (MW)": result.slack_power_mw,
})

A label of 0 means that every retained physical eigenvalue lies to the left of the configured stability margin; label 1 means unstable. `critical_real` is the later regression target. The label, `critical_imag` and `slack_power_mw` remain diagnostics, and none of these output quantities is an input to the neural network.